# Ingestion via Virtual Asset

**Persona:** Data Engineer

**Goal:** Demonstrate the ingestion pipeline using a virtual (external href) asset
without any upload flow:

1. Create a catalog and collection, apply `public_open_data` preset for routing
2. Register a **virtual** asset (`POST /assets/catalogs/{cat}/collections/{col}/virtual-assets`)
3. Trigger the ingestion task (`POST /processes/catalogs/{cat}/collections/{col}/processes/ingestion/execution`)
4. Poll task status via the OGC job `Location` header
5. Verify items appear via `GET /stac/catalogs/{cat}/collections/{col}/items`
6. Teardown

A **virtual asset** (`kind=virtual`) carries an external `href` — the platform
registers the reference without managing the bytes. Ingestion reads the file
directly from the external URI.

**Preset used:** `public_open_data` — composite preset, tier PLATFORM, catalog_scopable.
Defined at `modules/storage/presets/composites/public_open_data.py`.

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_SYSADMIN_TOKEN`

**Optional:** `INGESTION_SAMPLE_URI` — URI of a GeoJSON/CSV file the platform can read.
Defaults to a no-op placeholder that will produce a FAILED task on unsupported deployments;
substitute a real URI reachable by the platform to run ingestion end-to-end.


In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

SYSADMIN_TOKEN = os.environ.get("DYNASTORE_SYSADMIN_TOKEN", "")

# URI of a file the platform can open with Fiona/GDAL.
# Replace with a real GeoJSON or CSV path reachable from the platform.
INGESTION_URI = os.environ.get("INGESTION_SAMPLE_URI", "https://example.com/sample_points.geojson")

CATALOG_ID = f"nb06-{uuid.uuid4().hex[:8]}"
COLLECTION_ID = "crop-stations-virtual"
ASSET_ID = f"virtual-asset-{uuid.uuid4().hex[:8]}"

sysadmin_headers = {
    "Authorization": f"Bearer {SYSADMIN_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=sysadmin_headers, timeout=120.0)

print(f"BASE_URL      = {BASE_URL}")
print(f"CATALOG_ID    = {CATALOG_ID}")
print(f"COLLECTION_ID = {COLLECTION_ID}")
print(f"ASSET_ID      = {ASSET_ID}")
print(f"INGESTION_URI = {INGESTION_URI}")
print(f"sysadmin      = {'set' if SYSADMIN_TOKEN else 'NOT SET'}")

if not SYSADMIN_TOKEN:
    raise RuntimeError(
        "DYNASTORE_SYSADMIN_TOKEN is not set (and IDP_* auto-mint did not produce one). "
        "Set it before running — every step below needs sysadmin rights."
    )

## Step 1 — Create catalog, apply preset, create collection

Create the catalog, poll readiness, apply `public_open_data` preset to wire
routing, then create the collection.


In [ ]:
# Create catalog
r = client.post(
    "/stac/catalogs",
    content=json.dumps({
        "id": CATALOG_ID,
        "type": "Catalog",
        "title": "NB06 Virtual Asset Ingestion",
        "description": "Ephemeral catalog for the virtual-asset ingestion walkthrough.",
        "stac_version": "1.0.0",
    }),
)
print(f"Catalog create -> {r.status_code}")
assert r.status_code == 201, f"Expected 201, got {r.status_code}: {r.text}"


In [ ]:
# Poll readiness
for _attempt in range(30):
    rr = client.get(f"/admin/catalogs/{CATALOG_ID}")
    if rr.status_code == 200 and rr.json().get("provisioning_status") == "ready":
        print(f"Catalog ready after {_attempt} polls")
        break
    time.sleep(2)
else:
    print("WARNING: catalog still provisioning after 60s — proceeding anyway")


In [ ]:
# Apply public_open_data preset
r = client.post(
    f"/configs/catalogs/{CATALOG_ID}/presets/public_open_data",
    content="{}",
)
print(f"Preset apply -> {r.status_code}")
assert r.status_code in (200, 201, 204), (
    f"Preset apply expected 200/201/204, got {r.status_code}: {r.text}"
)
print("Preset 'public_open_data' applied")


In [ ]:
# Create collection
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps({
        "id": COLLECTION_ID,
        "type": "Collection",
        "stac_version": "1.0.0",
        "title": "Crop Monitoring Stations (Virtual)",
        "description": "Collection receiving features from a virtual asset ingestion.",
        "license": "proprietary",
        "extent": {
            "spatial": {"bbox": [[-180.0, -90.0, 180.0, 90.0]]},
            "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
        },
        "links": [],
    }),
)
print(f"Collection create -> {r.status_code}")
assert r.status_code == 201, f"Expected 201, got {r.status_code}: {r.text}"
print(f"Collection '{COLLECTION_ID}' created")


## Step 2 — Register a virtual asset

A virtual asset (`kind=virtual`) carries an external `href` — the platform
registers the reference without copying or managing the bytes. Ingestion reads
the file at runtime from the `href`.

Route: `POST /assets/catalogs/{cat}/collections/{col}/virtual-assets`

Body shape (`VirtualAssetCreate`): `asset_id`, `href`, optional `metadata`.


In [ ]:
r = client.post(
    f"/assets/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/virtual-assets",
    content=json.dumps({
        "asset_id": ASSET_ID,
        "href": INGESTION_URI,
        "asset_type": "ASSET",
        "metadata": {
            "description": "External GeoJSON/CSV for virtual ingestion demo.",
            "format": "application/geo+json",
        },
    }),
)
print(f"Virtual asset register -> {r.status_code}")
if r.status_code == 409:
    print("Asset already registered (409 OK — re-run safe)")
else:
    assert r.status_code in (200, 201), (
        f"Expected 200/201, got {r.status_code}: {r.text[:400]}"
    )
    print(f"Registered virtual asset: {r.json().get('asset_id', ASSET_ID)}")


## Step 3 — Trigger ingestion task

Submit the ingestion process. The `asset_id` in the request body tells the
ingestion engine which registered asset to open. The platform reads the file
at the virtual asset's `href`.

Route: `POST /processes/catalogs/{cat}/collections/{col}/processes/ingestion/execution`

The `Prefer: respond-async` header requests asynchronous execution; the
response is `201 Created` with a `Location` header for polling.


In [ ]:
exec_url = f"/processes/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/processes/ingestion/execution"

r = client.post(
    exec_url,
    content=json.dumps({
        "inputs": {
            "ingestion_request": {
                "asset": {"asset_id": ASSET_ID},
                "column_mapping": {
                    "attributes_source_type": "all",
                },
                "source_srid": 4326,
            }
        }
    }),
    headers={"Prefer": "respond-async", "Content-Type": "application/json"},
)
print(f"Ingestion submit -> {r.status_code}")
assert r.status_code in (200, 201), f"Expected 200/201, got {r.status_code}: {r.text[:400]}"

status_url = r.headers.get("Location", "")
job_info = r.json()
job_id = job_info.get("jobID", "")
print(f"Job ID       = {job_id}")
print(f"Status URL   = {status_url}")


## Step 4 — Poll task status

Poll the OGC job status URL until the job reaches a terminal state.
States: `accepted` -> `running` -> `successful` or `failed`.


In [ ]:
TERMINAL = {"successful", "failed", "dismissed"}
final_status = None

if status_url:
    # Strip base URL prefix so httpx can route the relative path
    rel_path = status_url
    if status_url.startswith(BASE_URL):
        rel_path = status_url[len(BASE_URL):]

    for elapsed in range(0, 120, 3):
        rp = client.get(rel_path)
        if rp.status_code != 200:
            print(f"  [{elapsed}s] poll error {rp.status_code}")
            break
        info = rp.json()
        status = info.get("status", "unknown")
        progress = info.get("progress", 0)
        print(f"  [{elapsed}s] status={status} progress={progress}%")
        if status in TERMINAL:
            final_status = status
            break
        time.sleep(3)
    else:
        print("Timeout: job did not reach terminal state within 120s")
else:
    print("No Location header returned — synchronous execution assumed")
    final_status = "successful"

print(f"\nFinal status: {final_status}")
if final_status == "failed":
    print("NOTE: ingestion FAILED — likely because the URI is a placeholder.")
    print("Set INGESTION_SAMPLE_URI to a real file reachable by the platform.")


## Step 5 — Verify items via STAC

After a successful ingestion, items should appear in the collection.
A failed ingestion (placeholder URI) will return 0 items, which is expected
in demo environments.


In [ ]:
r = client.get(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    params={"limit": 10},
)
print(f"GET /items -> {r.status_code}")
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text[:400]}"

result = r.json()
features = result.get("features", [])
returned = result.get("context", {}).get("returned", len(features))
print(f"Items returned: {returned}")

for feat in features[:5]:
    print(f"  - {feat.get('id')}")

if final_status == "successful":
    assert returned > 0, "Expected items after successful ingestion — none found"
    print("Ingestion verified: items present via STAC.")
else:
    print(f"Ingestion status was '{final_status}' — item count may be 0 (expected for placeholder URI).")


## Teardown

Force-delete the test catalog. All collections, assets, items, and the
PostgreSQL schema for this catalog are removed.


In [ ]:
r = client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"})
print(f"Teardown -> {r.status_code}")
assert r.status_code == 204, f"Expected 204, got {r.status_code}: {r.text}"
print(f"Catalog {CATALOG_ID} deleted.")
client.close()
